In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train = pd.read_csv("../data/processed/train.csv")
validation = pd.read_csv("../data/processed/validation.csv")
test = pd.read_csv("../data/processed/test.csv")

In [ ]:
numeric_cols = [
    'RevolvingUtilizationOfUnsecuredLines',
    'age',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'DebtRatio',
    'MonthlyIncome',
    'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate',
    'NumberRealEstateLoansOrLines',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfDependents'
]

outlier_summary = train[numeric_cols].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
        0.999
    ]
).T

outlier_summary

In [ ]:
# Ratio max to 99th percentile
outlier_summary["max_to_p99_ratio"] = (
    outlier_summary["max"]
    /
    outlier_summary["99%"]
)

outlier_summary[
    ["99%", "max", "max_to_p99_ratio"]
].sort_values(
    by="max_to_p99_ratio",
    ascending=False
)

# IQR Method

In [ ]:
def iqr_outliers(df, variable):

    q1 = df[variable].quantile(0.25)
    q3 = df[variable].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = (
        (df[variable] < lower)
        |
        (df[variable] > upper)
    )

    return outliers.sum()

In [ ]:
for col in numeric_cols:

    print(
        col,
        iqr_outliers(train, col)
    )

# z score

In [ ]:
# standard deviation method
from scipy.stats import zscore

z_scores = zscore(
    train["MonthlyIncome"].dropna()
)

(sum(abs(z_scores) > 3))

# Plot

In [ ]:
def plot_box(df, variable):

    plt.figure(figsize=(10,2))

    plt.boxplot(
        df[variable].dropna(),
        vert=False
    )

    plt.title(variable)

    plt.show()

In [ ]:
plot_box(train, "DebtRatio")
plot_box(train, "MonthlyIncome")
plot_box(train, "RevolvingUtilizationOfUnsecuredLines")

In [ ]:
(train["age"] == 0).sum()


## 5.1 Outlier Identification

### Key Findings

The analysis of percentile statistics revealed substantial right-skewness and extreme values in several variables.

The most pronounced outliers were observed in:

| Variable | P99 | Max | Max / P99 |
|----------|---------:|---------:|---------:|
| RevolvingUtilizationOfUnsecuredLines | 1.10 | 50,708 | 46,305 |
| MonthlyIncome | 25,000 | 3,008,750 | 120 |
| DebtRatio | 4,948 | 329,664 | 67 |

In addition, delinquency-related variables exhibited extreme values:

- `NumberOfTime30-59DaysPastDueNotWorse`
- `NumberOfTimes90DaysLate`
- `NumberOfTime60-89DaysPastDueNotWorse`

where the maximum value equals **98**, despite the 99th percentile ranging between **2 and 4**.

`Age` exhibited the lowest level of outlier influence, with a maximum-to-P99 ratio of only **1.25**.

---

## 5.2 Business Validation

The identified extreme values were reviewed from a business perspective.

### RevolvingUtilizationOfUnsecuredLines

Values substantially above 100% may occur due to reporting inconsistencies, timing differences, or special credit line situations.

### MonthlyIncome

Extremely high income values may represent genuine high-income customers rather than data errors.

### DebtRatio

Very large debt ratios may occur when customers report very low income while maintaining significant outstanding debt obligations.

### Delinquency Variables

The value **98** appears repeatedly across delinquency-related variables and is likely to represent a special coded value rather than an actual number of delinquency events.

This observation will be investigated further during the binning stage.

### Age

The presence of `Age = 0` represents a data quality issue and will be treated as a missing or special value during later preprocessing.

---

## 5.3 Treatment Decision

No outlier treatment will be applied at this stage.

The following approaches were considered:

- Winsorization
- Capping
- Flooring
- Log Transformation
- Observation Removal

However, since the subsequent modelling methodology relies on **Optimal Binning** and **Weight of Evidence (WoE)** transformation, extreme values can be handled naturally through binning without modifying the original observations.

Therefore:

- Original variable values will be retained.
- Outliers will be addressed during OptBinning.
- Special values (e.g. `Age = 0` and potentially delinquency value `98`) will be reviewed separately.

---

## Conclusion

Several variables exhibit significant outlier behaviour, particularly:

- `RevolvingUtilizationOfUnsecuredLines`
- `DebtRatio`
- `MonthlyIncome`

Despite the presence of extreme values, no pre-binning outlier treatment is considered necessary. The selected scorecard methodology (**OptBinning + WoE transformation**) is expected to provide a robust treatment of these observations while preserving potentially valuable predictive information.